# A probe sphere travelling through a galaxy

A small sphere, about as wide as the gas disc is thick, is carried once around the galaxy. Only the
gas **inside the sphere** is ever shown. The camera never moves: it sits at 60 degrees, so the
orbit projects to an ellipse and you watch the probe run around it.

Every couple of seconds the movie switches what it is measuring inside that same sphere: surface
density, line-of-sight velocity, velocity dispersion, temperature, pressure, and finally the stars.
The name of the quantity is on screen the whole time.

The point is that **one region selects, and many quantities answer**. The sphere is a value you
move; what you ask of the gas inside it is a separate decision.

**To run it on your own simulation, change two things in the next cell**: the path and the output
number.

> This is a gallery recipe. It is not part of Mera's test suite, and it is not executed when the
> documentation is built, so its outputs are not stored here. Run it and it will produce them.

**What you need:** a RAMSES output or a converted MERA file with gas and particles, CairoMakie,
ColorSchemes, and ffmpeg for the last step.

![The probe running around the galaxy](media/probe_preview.gif)

*Reduced preview. Full resolution in [`media/probe_movie.mp4`](media/probe_movie.mp4).*

| | |
|---|---|
| **Author** | Manuel Behrendt |
| **Contact** | [github.com/ManuelBehrendt](https://github.com/ManuelBehrendt) |
| **Reads** | RAMSES |
| **Provenance** | `Mera v1.8.0 \| AV05CD/output_00390 \| 580.05 Myr \| L=48.0 ndim=3 lmin=6 lmax=12` |
| **Status** | contributed 2026-09-16 |

## The environment

This recipe carries its own `Project.toml`. Running the cell below uses the package versions it was
written against. See
[Reproducibility](https://manuelbehrendt.github.io/Mera.jl/stable/reproducibility/).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## 1. The two lines you change

In [ ]:
using Mera, Printf, ColorSchemes
using CairoMakie
CairoMakie.activate!(type = "png")

# ---- change these two ----------------------------------------------------------------
const SIM    = "/path/to/your/simulation"   # a RAMSES output folder, or a MERA-file folder
const OUTPUT = 390                          # the snapshot number
# --------------------------------------------------------------------------------------

const R_PATH  = 4.5      # kpc, radius of the circle the probe is carried around
const R_PROBE = 1.5      # kpc, radius of the probe itself: about the half-thickness of the disc
const INC, AZ = 60.0, 30.0    # degrees: the camera, fixed for the whole movie

const PX       = 12.0    # pc, pixel size. The finest cells in this run are
                         # 11.7 pc, so this is about as sharp as the data allows
const SD_FLOOR = 0.01    # Msol/pc^2, below this there is no gas worth colouring
const NFRAMES  = 240     # 12 seconds at 20 fps; try 12 first

# the frame, in kpc. The orbit is a circle in the disc, so at 60 degrees it projects to an
# ellipse: wider than it is tall, and the frame follows that.
const FRAME_X, FRAME_Y = 6.4, 3.9

## 2. Read a slab, and keep only the ring the probe will visit

Two savings, both worth copying.

The **read** is limited to a slab that contains the orbit, so the rest of the box never enters
memory.

Then the **ring**: the probe only ever visits an annulus of radius `R_PATH ± R_PROBE`, so that
annulus is selected once, with region algebra, and every frame works against it instead of against
the whole slab. On the run this was written for, that is well under half the cells, and the per-frame
cost falls with it. A difference of one cylinder minus another, computed once.

In [ ]:
(; hydro, particles, info) = loadall(SIM, OUTPUT;
                                    components = [:hydro, :particles],
                                    xrange = [-(R_PATH + 2R_PROBE), R_PATH + 2R_PROBE],
                                    yrange = [-(R_PATH + 2R_PROBE), R_PATH + 2R_PROBE],
                                    zrange = [-2R_PROBE, 2R_PROBE],
                                    center = [:bc], range_unit = :kpc)

ring = Mera.Cylinder(R_PATH + R_PROBE, R_PROBE; center = [:bc], range_unit = :kpc) \
       Mera.Cylinder(R_PATH - R_PROBE, R_PROBE; center = [:bc], range_unit = :kpc)

gas_ring  = subregion(hydro,     ring, verbose = false)
star_ring = subregion(particles, ring, verbose = false)

const BC = hydro.boxlen * hydro.info.scale.kpc / 2   # the box centre, in kpc

@printf("slab %d cells -> ring %d cells (%.0f%%), %d stars\n",
        length(hydro.data), length(gas_ring.data),
        100 * length(gas_ring.data) / length(hydro.data), length(star_ring.data))

**`Mera.Cylinder`, not `Cylinder`.** Makie exports geometry types of its own, and `Sphere` in
particular collides: with both packages loaded, a bare `Sphere` is ambiguous and Julia raises an
`UndefVarError`. Writing `Mera.Sphere` and `Mera.Cylinder` says which one you mean and costs
nothing.

## 3. The edge of the sphere is cut, not stepped

A sphere drawn through a grid of cubes has a ragged boundary unless something is done about it.
`subregion` splits the boundary cells by default (`split = true`): a cell half inside the probe
keeps half of itself, and carries that share as `:fraction`.

That matters twice over. The **mass** is the exact mass inside the sphere rather than the mass of
whichever cells happened to have their centres inside. And the **picture** is smoother, because a
boundary cell deposits its reduced share rather than its whole contents, so the rim falls off
instead of stepping.

The check below is the one worth keeping: the projected map totals the same mass that `msum`
reports, which is what tells you the fractions survived all the way through to the image.

In [ ]:
probe0 = Mera.Sphere(R_PROBE; center = [BC + R_PATH, BC, BC], range_unit = :kpc)

exact = subregion(gas_ring, probe0, verbose = false)                 # split = true by default
whole = subregion(gas_ring, probe0, split = false, verbose = false)  # centre-inside test

pm = projection(exact, :mass, :Msol; inclination = INC, azimuth = AZ, pxsize = [PX, :pc],
                center = [:bc], verbose = false, show_progress = false)

@printf("cells kept     exact %d   whole-cell %d\n", length(exact.data), length(whole.data))
@printf("mass in probe  exact %.5e   whole-cell %.5e   (%.3f%% high)\n",
        msum(exact, :Msol), msum(whole, :Msol),
        100 * (msum(whole, :Msol) / msum(exact, :Msol) - 1))
@printf("projected map totals %.5e  ->  ratio to msum %.5f\n",
        sum(filter(isfinite, pm.maps[:mass])), sum(filter(isfinite, pm.maps[:mass])) / msum(exact, :Msol))

## 4. What the probe measures

One row per view. Each says what to ask for, in which unit, what to call it on screen, which
colour map, the range to stretch over, whether that range is in log, and which component to ask.

The ranges are fixed for the whole movie, deliberately. If each frame rescaled to its own
contents, the colours would pulse and nothing could be compared with anything.

In [ ]:
# quantity, unit, on-screen name, colour map, (lo, hi), log?, component
const VIEWS = [
    (:sd,   :Msol_pc2, "gas surface density",    ColorSchemes.magma,   (-1.5, 2.2),     true,  :gas),
    (:vlos, :km_s,     "line-of-sight velocity", ColorSchemes.vik,     (-220.0, 220.0), false, :gas),
    (:σlos, :km_s,     "velocity dispersion",    ColorSchemes.viridis, (0.0, 60.0),     false, :gas),
    (:T,    :K,        "temperature",            ColorSchemes.inferno, (2.5, 6.5),      true,  :gas),
    (:p,    :p_kB,     "pressure",               ColorSchemes.plasma,  (1.0, 5.0),      true,  :gas),
    (:sd,   :Msol_pc2, "stars",                  ColorSchemes.bone,    (-1.5, 1.0),     true,  :stars),
]

`vik` is one of Crameri's scientific colour maps, built for diverging data and for colour vision
deficiency. A signed quantity needs a diverging map: a sequential one hides the sign, and here the
sign is the content, which side of the gas is coming towards you.

## 5. Two helpers

`stretch` maps a quantity onto 0..1 between fixed limits, in log or linear, and turns anything
outside into `NaN` so it is drawn as background.

`regrid` resamples a frame onto one canvas that is the same for every frame. Without it the frame
would be whatever the probe's own projection happened to span, and the picture would jump around as
the probe moved.

In [ ]:
"""Map onto 0..1 between fixed limits, in log or linear. Anything unusable becomes NaN."""
function stretch(a, lo, hi, uselog)
    out = similar(a, Float64)          # similar(), not a comprehension: keep the 2-D shape
    @inbounds for i in eachindex(a)
        v = a[i]
        out[i] = !isfinite(v) ? NaN :
                 uselog ? (v > 0 ? clamp((log10(v) - lo) / (hi - lo), 0, 1) : NaN) :
                          clamp((v - lo) / (hi - lo), 0, 1)
    end
    return out
end

"""Resample `m`, which spans `ext = [x0,x1,y0,y1]`, onto a fixed grid covering `dst`."""
function regrid(m, ext, nx, ny, dst)
    out = fill(NaN, nx, ny)
    sx, sy = size(m)
    x0s, x1s, y0s, y1s = ext
    x0d, x1d, y0d, y1d = dst
    @inbounds for j in 1:ny, i in 1:nx
        x = x0d + (i - 0.5) * (x1d - x0d) / nx
        y = y0d + (j - 0.5) * (y1d - y0d) / ny
        (x < x0s || x > x1s || y < y0s || y > y1s) && continue
        is = clamp(ceil(Int, (x - x0s) / (x1s - x0s) * sx), 1, sx)
        js = clamp(ceil(Int, (y - y0s) / (y1s - y0s) * sy), 1, sy)
        out[i, j] = m[is, js]
    end
    return out
end

const CANVAS = (ext = [-FRAME_X, FRAME_X, -FRAME_Y, FRAME_Y],
                nx  = round(Int, 2FRAME_X * 1000 / PX),
                ny  = round(Int, 2FRAME_Y * 1000 / PX))

## 6. The frames

For each frame: put the probe at its place on the circle, select the gas inside it, project, and
draw.

`Mera.Sphere` takes its centre in the same unit as `range_unit`, measured from the corner of the
box, which is why the box centre is added to the orbit offset.

**One subtlety about empty pixels.** A log quantity masks itself, because zero is not positive and
so becomes `NaN`. A linear one does not: outside the probe the projected map is zero, and zero on a
diverging scale is a perfectly good colour, so the sphere ends up sitting in a pale square. The
surface density says where there is gas at all, so it is used to mask the linear quantities.

In [ ]:
turn(k)  = 2π * k / NFRAMES
view_at(k) = VIEWS[min(length(VIEWS), floor(Int, (k / NFRAMES) * length(VIEWS)) + 1)]

const AXW = 1240
const AXH = round(Int, AXW * FRAME_Y / FRAME_X)

frames = joinpath("media", "probe_frames"); mkpath(frames)
trail  = Tuple{Float64,Float64}[]
t0 = time()

for k in 0:(NFRAMES - 1)
    theta = turn(k)
    q, u, name, cs, lims, uselog, comp = view_at(k)

    probe = Mera.Sphere(R_PROBE;
                        center = [BC + R_PATH * cos(theta), BC + R_PATH * sin(theta), BC],
                        range_unit = :kpc)
    sel = subregion(comp === :gas ? gas_ring : star_ring, probe, verbose = false)

    kw  = (inclination = INC, azimuth = AZ, pxsize = [PX, :pc], center = [:bc],
           verbose = false, show_progress = false)
    pr  = projection(sel, q, u; kw...)
    raw = pr.maps[q]
    if !uselog                                   # see the note above about empty pixels
        sd  = projection(sel, :sd, :Msol_pc2; kw...).maps[:sd]
        raw = copy(raw); raw[sd .< SD_FLOOR] .= NaN
    end

    c   = pr.cextent                             # the window, measured from `center`
    img = regrid(stretch(raw, lims[1], lims[2], uselog), c, CANVAS.nx, CANVAS.ny, CANVAS.ext)

    fig = Figure(size = (AXW, AXH + 86), backgroundcolor = :black, figure_padding = (2,2,2,2))
    ax  = Axis(fig[1, 1]; width = AXW, height = AXH, backgroundcolor = :black)
    hidedecorations!(ax); hidespines!(ax)
    image!(ax, CANVAS.ext[1] .. CANVAS.ext[2], CANVAS.ext[3] .. CANVAS.ext[4], img;
           colormap = cgrad(cs), colorrange = (0, 1), nan_color = :black)

    cx, cy = (c[1] + c[2]) / 2, (c[3] + c[4]) / 2        # the probe, as projected
    push!(trail, (cx, cy))
    length(trail) > 1 && lines!(ax, first.(trail), last.(trail);
                                color = (:white, 0.18), linewidth = 1)
    lines!(ax, cx .+ R_PROBE .* cos.(range(0, 2π, 180)),
               cy .+ R_PROBE .* sin.(range(0, 2π, 180)); color = (:white, 0.30), linewidth = 1)

    Label(fig[0, 1], name; color = :white, fontsize = 30, font = :bold, padding = (0,0,2,6))
    Label(fig[2, 1], @sprintf("%s  ·  probe r = %.1f kpc on a %.1f kpc orbit  ·  %.0f° inclination",
                              string(u), R_PROBE, R_PATH, INC);
          color = RGBf(0.6, 0.62, 0.68), fontsize = 15, padding = (0,0,4,2))
    rowgap!(fig.layout, 2); resize_to_layout!(fig)
    save(joinpath(frames, @sprintf("f_%04d.png", k)), fig)

    el = time() - t0
    @printf("  %3d/%d  %-22s θ=%5.1f°   %5.1f s, ~%4.1f min left\n",
            k + 1, NFRAMES, name, rad2deg(theta), el, (el/(k+1)*(NFRAMES-k-1))/60)
end

**Why the axis box is pinned with `width` and `height`.** Left to itself, the axis is sized by the
layout and the result drifts with the labels, which for a movie means the scale changes between
frames. Fixing the box in pixels, with the canvas fixed in kpc, makes every frame identical in
geometry, which is the only way a sequence of them reads as motion rather than as a slideshow.

## 7. Assemble

In [ ]:
const FFMPEG = "ffmpeg"     # or an absolute path to the binary
pat = joinpath(frames, "f_%04d.png")

run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat
     -vf "scale=1800:-2:flags=lanczos" -c:v libx264 -pix_fmt yuv420p -crf 20
     -movflags +faststart media/probe_movie.mp4`)

run(`$FFMPEG -y -loglevel error -i $pat
     -vf "fps=12,scale=560:-2:flags=lanczos,palettegen=max_colors=96"
     $(joinpath(frames, "palette.png"))`)
run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat -i $(joinpath(frames, "palette.png"))
     -lavfi "fps=12,scale=560:-2:flags=lanczos[x];[x][1:v]paletteuse=dither=none"
     -loop 0 media/probe_preview.gif`)

@printf("mp4 %.1f MB, gif %.1f MB\n",
        filesize("media/probe_movie.mp4")/1e6, filesize("media/probe_preview.gif")/1e6)

## The thumbnail

The card wants the probe itself. A frame lifted straight out of the movie has the sphere small and
off to one side, with most of the card empty, because the movie frame is wide enough to hold the
whole orbit. The same projection drawn on its own extent fills the picture instead.

In [ ]:
pp = projection(subregion(gas_ring, probe0, verbose = false), :sd, :Msol_pc2;
                inclination = INC, azimuth = AZ, pxsize = [PX, :pc], center = [:bc],
                verbose = false, show_progress = false)

fig = Figure(size = (1400, 790), backgroundcolor = :black, figure_padding = 0)
ax  = Axis(fig[1, 1]; aspect = DataAspect(), backgroundcolor = :black)
hidedecorations!(ax); hidespines!(ax)
c = pp.cextent
image!(ax, c[1] .. c[2], c[3] .. c[4], stretch(pp.maps[:sd], -1.5, 2.2, true);
       colormap = cgrad(ColorSchemes.magma), colorrange = (0, 1), nan_color = :black)
save("media/probe_hero.png", fig)

makethumb("media/probe_hero.png", "media/probe_thumb.png"; mode = :crop)

## Attribution: run this once and paste the result

In [ ]:
println("Reads        ", info.simcode)
println("Mera         ", mera_build())
println("Provenance   ", provenance_string(hydro))

## Making it yours

- **A different path.** The probe follows a circle only because `turn(k)` says so. Give it a radial
  plunge, a straight line across the disc, or a rise out of the plane; only the centre passed to
  `Mera.Sphere` changes. Remember to widen the ring in section 2 to whatever the new path visits.
- **A different probe.** `Mera.Sphere` can be a `Mera.Cuboid`, or a sphere with a hole in it. Any
  region works, and the rest of the recipe does not change.
- **Other quantities.** Add a row to `VIEWS`. Anything `projection` accepts will do, including a
  field you registered yourself with `add_field`.
- **Numbers instead of pictures.** The same loop with `msum`, `wstat` or `center_of_mass` in place
  of the projection gives you the quantity as a function of position along the path, which is
  usually the measurement the picture is standing in for.
- **Follow the probe.** Keeping the camera fixed shows where the probe is. Centring the view on the
  probe instead shows what is inside it in more detail, at the cost of losing the sense of place.